# Problem Set 9: Project Genesis – The Autonomous Engineer

In Module 2, you unlocked the power of high-performance physics engines using JAX. In Module 3, you connected these simulators to cognitive models (local Gemma 4 models). But until now, the human has always been the bottleneck—analyzing the graphs, adjusting parameters, and clicking 'Run' again.

Today, we cross the threshold of autonomy. You will build the **Silicon Cartographer**. Your mission is to delegate the entire exploration of a JAX-accelerated Mandelbrot simulation to an agentic loop. The agent will act as an experimental scientist: starting from a global view, it will dynamically zoom in, analyze boundaries, calculate entropy, and refine its search coordinates to locate specific fractal details (like "Seahorse Valley").

We will implement this in three distinct phases:
1. **Manual Prompting (Pure Model)**: Experience the coordination overhead of manual human-in-the-loop parameter selection.
2. **Native Tool Calling (Model + Tools)**: Implement declarative Function Calling schemas and construct an automated execution pipeline.
3. **Capsule Packaging: The Gemma-Skill (Model + Tools + Skills)**: Structure the agent architecture into a modular, shareable, and self-contained Gemma-Skill unit.

Let's build!

## Exercise 1: The Manual Cartographer (Pure Model)

We start by running a JAX-accelerated Mandelbrot simulator. In this exercise, the agent only reasons in natural language, and the human is responsible for copying coordinates and running the code.

Below is the JAX-accelerated Mandelbrot simulation code that calculates the fractal and returns complexity metrics (Shannon Entropy of escape times and boundary pixel ratio).

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# JAX Mandelbrot Core Kernel
@jax.jit
def mandelbrot_kernel(c, max_iters):
    def body_fn(val):
        z, count, active = val
        next_z = z**2 + c
        next_active = active & (jnp.abs(next_z) <= 2.0)
        next_count = jnp.where(next_active, count + 1, count)
        return next_z, next_count, next_active

    def cond_fn(val):
        _, count, active = val
        return jnp.any(active) & (jnp.min(count) < max_iters)

    z = jnp.zeros_like(c)
    count = jnp.zeros_like(c, dtype=jnp.int32)
    active = jnp.ones_like(c, dtype=jnp.bool_)

    _, final_counts, _ = jax.lax.while_loop(cond_fn, body_fn, (z, count, active))
    return final_counts

def run_simulation(center_real, center_imag, zoom, resolution=400, max_iterations=500):
    # Map resolution coordinates to complex plane
    width, height = resolution, resolution
    r = jnp.linspace(center_real - 1.5 / zoom, center_real + 1.5 / zoom, width)
    i = jnp.linspace(center_imag - 1.5 / zoom, center_imag + 1.5 / zoom, height)
    R, I = jnp.meshgrid(r, i)
    C = R + 1j * I

    # Execute high-performance calculation on accelerator
    counts = mandelbrot_kernel(C.flatten(), max_iterations)
    counts = counts.reshape((height, width))

    # Calculate Shannon Entropy of escape times
    hist, _ = jnp.histogram(counts, bins=20)
    hist_prob = hist / jnp.sum(hist)
    hist_prob = jnp.where(hist_prob > 0, hist_prob, 1.0)  # Avoid log(0)
    entropy = -jnp.sum(hist_prob * jnp.log(hist_prob))

    # Calculate Boundary Complexity (ratio of boundary pixels)
    boundary_pixels = jnp.sum((counts > 0) & (counts < max_iterations))
    boundary_ratio = boundary_pixels / (width * height)

    return counts, {
        "entropy": float(entropy),
        "boundary_complexity": float(boundary_ratio),
        "center_real": float(center_real),
        "center_imag": float(center_imag),
        "zoom": float(zoom),
        "max_iterations": int(max_iterations)
    }

In [ ]:
# Test execution at global view
counts, metrics = run_simulation(center_real=-0.5, center_imag=0.0, zoom=1.5)
print("Simulation Metrics:", metrics)

plt.figure(figsize=(6, 6))
plt.imshow(counts, cmap='twilight_shifted', extent=[-2, 1, -1.5, 1.5])
plt.colorbar(label='Iterations until escape')
plt.title('Mandelbrot Global View (JAX)')
plt.show()

### Task:
1. Write a prompt to your LLM (using the Gemini API or a local Gemma model).
2. Feed it the `metrics` dictionary obtained above.
3. Ask the model to suggest coordinates to navigate toward the **"Seahorse Valley"** ($c \approx -0.7436, 0.1318i$) and increase zoom.
4. Manually run the simulation with the suggested coordinates, extract the new metrics, and pass them back to the model for another iteration. Do this for 3 steps.

**Question:** What are the latency and usability limitations of this manual human-in-the-loop coordination compared to automated loops?

*Enter your answer here:*

## Exercise 2: Closed-Loop Tool Calling (Model + Tools)

To automate the search loop, you will define the simulation as a **Tool** that the model can invoke autonomously using the Gemini Function Calling interface. 

### Instructions:
1. Initialize your Gemini API Client (`google-genai` SDK).
2. Register the `run_simulation` function as a tool using the `tools` parameter in client configuration.
3. Write a Python ReAct execution loop that intercepts the model's `function_call` outputs, calls the local JAX simulation, returns the result as a `tool_response` (Observation), and continues until the model converges on Seahorse Valley (zoom >= 15,000x).

In [ ]:
# Install the modern Google GenAI SDK if needed
# !pip install google-genai python-dotenv

import os
from google import genai
from google.genai import types

# Set your API Key
# os.environ["GEMINI_API_KEY"] = "your-api-key"

# Define the tool wrapper that conforms to python typing
def simulate_mandelbrot(center_real: float, center_imag: float, zoom: float, max_iterations: int = 500) -> dict:
    """
    Runs a JAX-accelerated Mandelbrot simulation on the specified center coordinates and zoom factor.
    Returns visual complexity and Shannon entropy metrics.
    """
    _, metrics = run_simulation(center_real, center_imag, zoom, max_iterations=max_iterations)
    return metrics

# TODO: Implement the ReAct loop below
def run_autonomous_agent(target_description):
    # 1. Initialize GenAI client
    # client = genai.Client()
    
    # 2. Setup the tool registration list
    # tools = [simulate_mandelbrot]
    
    # 3. Write a loop to handle the Thought-Action-Observation sequence
    # For each step:
    #   a. Send conversation history to the model with the registered tool list.
    #   b. If model requests a function call, execute simulate_mandelbrot with the arguments.
    #   c. Append the function response (Observation) back to the history.
    #   d. Print the Thought and Tool call parameters.
    #   e. Break when the model decides to stop (generates text instead of function call).
    pass

## Exercise 3: Capsule Packaging: The Gemma-Skill (Model + Tools + Skills)

To make your autonomous engineer modular, reusable, and compatible with larger ecosystems, package it into a standard **Gemma-Skill** capsule following the `google-gemma/gemma-skills` specification.

### Instructions:
1. Create a local folder called `skills/mandelbrot_explorer`.
2. Write a `SKILL.md` file. It must start with a YAML frontmatter declaring the skill's name and description, followed by Markdown system prompt instructions explaining how to search for high-complexity boundaries using the simulation feedback.
3. Save the JSON tool schema of `simulate_mandelbrot` under `skills/mandelbrot_explorer/tools/mandelbrot_schema.json`.
4. Save your JAX solver code under `skills/mandelbrot_explorer/scripts/mandelbrot_solver.py`.
5. Write a python script below that acts as a bootstrap loader: it dynamically loads the skill folder, registers the tool schemas, loads the instructions as system prompts, and launches the autonomous optimizer.

In [ ]:
import yaml
import json

class GemmaSkillLoader:
    def __init__(self, skill_dir):
        self.skill_dir = skill_dir
        self.instructions = ""
        self.metadata = {}
        self.schemas = []
        self.load_skill()
        
    def load_skill(self):
        # TODO: Implement loader logic
        # 1. Parse YAML frontmatter and Markdown body from SKILL.md
        # 2. Load JSON schemas from the tools/ directory
        pass

# TODO: Boot the agent using your dynamic skill loader and demonstrate the autonomous optimization loop
print("Loading Gemma-Skill...")

**Question:** How does packaging capabilities into self-contained Gemma-Skills improve system maintainability in multi-agent environments?

*Enter your answer here:*